# NLBSE'26 Code Comment Classification
**Improved approach over SetFit baseline**

Strategy: Replace `paraphrase-MiniLM-L6-v2` with `microsoft/codebert-base` fine-tuned with standard multi-label classification.
CodeBERT is pre-trained on code + natural language, so it understands code comments much better than a general-purpose sentence transformer.

Scoring formula (same as baseline notebook):
```
score = 0.6 × avg_F1 + 0.2 × speed_bonus + 0.2 × efficiency_bonus
```

## Step 1 — Install dependencies

In [1]:
!pip install -q datasets==3.6.0 transformers>=4.40.0 torch scikit-learn evaluate accelerate

## Step 2 — Imports and config

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from torch import nn
from torch.utils.data import DataLoader, Dataset as TorchDataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

langs = ['java', 'python', 'pharo']
label_names = {
    'java':   ['summary', 'Ownership', 'Expand', 'usage', 'Pointer', 'deprecation', 'rational'],
    'python': ['Usage', 'Parameters', 'DevelopmentNotes', 'Expand', 'Summary'],
    'pharo':  ['Keyimplementationpoints', 'Example', 'Responsibilities', 'Intent', 'Keymessages', 'Collaborators']
}

MODEL_NAME = 'microsoft/codebert-base'
MAX_LEN    = 128
EPOCHS     = 7
BATCH_SIZE = 16
LR         = 2e-5
CLASS_WEIGHT_SCALE = 10.0

Using device: cuda


## Step 3 — Load dataset

In [ ]:
ds = load_dataset('NLBSE/nlbse26-code-comment-classification')
print(ds)

DatasetDict({
    java_train: Dataset({
        features: ['index', 'class', 'comment_sentence', 'partition', 'combo', 'labels'],
        num_rows: 5394
    })
    java_test: Dataset({
        features: ['index', 'class', 'comment_sentence', 'partition', 'combo', 'labels'],
        num_rows: 1201
    })
    pharo_train: Dataset({
        features: ['index', 'class', 'comment_sentence', 'partition', 'combo', 'labels'],
        num_rows: 900
    })
    pharo_test: Dataset({
        features: ['index', 'class', 'comment_sentence', 'partition', 'combo', 'labels'],
        num_rows: 208
    })
    python_train: Dataset({
        features: ['index', 'class', 'comment_sentence', 'partition', 'combo', 'labels'],
        num_rows: 1368
    })
    python_test: Dataset({
        features: ['index', 'class', 'comment_sentence', 'partition', 'combo', 'labels'],
        num_rows: 290
    })
})


## Step 4 — Dataset class and helpers

In [ ]:
class CommentDataset(TorchDataset):
    """Wraps an HF split for PyTorch DataLoader."""
    def __init__(self, hf_split, tokenizer, max_len):
        self.texts  = hf_split['combo']   # class_name + sentence (same as baseline)
        self.labels = hf_split['labels']  # list of binary lists
        self.tok    = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.float)
        }

## Step 5 — Model definition

In [ ]:
class CodeBERTClassifier(nn.Module):
    """CodeBERT encoder + linear head for multi-label classification."""
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size  # 768 for codebert-base
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # [CLS] token
        return self.classifier(self.dropout(cls))

## Step 6 — Training loop

In [ ]:
def train_model(lang, ds, tokenizer):
    num_labels = len(label_names[lang])

    train_loader = DataLoader(
        CommentDataset(ds[f'{lang}_train'], tokenizer, MAX_LEN),
        batch_size=BATCH_SIZE, shuffle=True
    )

    model = CodeBERTClassifier(MODEL_NAME, num_labels).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    # Weighted loss
    all_labels = torch.tensor(ds[f'{lang}_train']['labels'], dtype=torch.float)
    pos_count = all_labels.sum(dim=0).clamp(min=1)
    neg_count = len(all_labels) - pos_count
    pos_weight = (neg_count / pos_count).clamp(max=CLASS_WEIGHT_SCALE).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for batch in tqdm(train_loader, desc=f'{lang} epoch {epoch+1}/{EPOCHS}'):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f'  [{lang}] epoch {epoch+1} loss: {total_loss/len(train_loader):.4f}')

    return model

## Step 7 — Train all three models

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
trained_models = {}

for lang in langs:
    print(f'\n=== Training {lang} ===')
    trained_models[lang] = train_model(lang, ds, tokenizer)

print('\nAll models trained!')


=== Training java ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

java epoch 1/7:   0%|          | 0/338 [00:00<?, ?it/s]

  [java] epoch 1 loss: 0.4431


java epoch 2/7:   0%|          | 0/338 [00:00<?, ?it/s]

...

  [pharo] epoch 5 loss: 0.2790


pharo epoch 6/7:   0%|          | 0/57 [00:00<?, ?it/s]

  [pharo] epoch 6 loss: 0.2036


pharo epoch 7/7:   0%|          | 0/57 [00:00<?, ?it/s]

  [pharo] epoch 7 loss: 0.1624

All models trained!


## Step 8 — Evaluation (same scoring as baseline notebook)

In [ ]:
def evaluate_model(lang, model, ds, tokenizer):
    test_loader = DataLoader(
        CommentDataset(ds[f'{lang}_test'], tokenizer, MAX_LEN),
        batch_size=64, shuffle=False
    )

    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            logits = model(
                batch['input_ids'].to(device),
                batch['attention_mask'].to(device)
            )
            preds = (torch.sigmoid(logits) > 0.5).cpu().numpy().astype(int)
            all_preds.append(preds)
            all_labels.append(batch['labels'].numpy().astype(int))

    y_pred = np.vstack(all_preds).T   # shape: (num_labels, num_samples)
    y_true = np.vstack(all_labels).T

    scores = []
    for i, cat in enumerate(label_names[lang]):
        tp = int(np.sum((y_true[i] == 1) & (y_pred[i] == 1)))
        fp = int(np.sum((y_true[i] == 0) & (y_pred[i] == 1)))
        fn = int(np.sum((y_true[i] == 1) & (y_pred[i] == 0)))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2*tp) / (2*tp + fp + fn) if (2*tp + fp + fn) > 0 else 0.0
        scores.append({'lang': lang, 'category': cat,
                       'precision': precision, 'recall': recall, 'f1': f1})
    return scores

In [ ]:
# Measure runtime (averaged over 10 runs, same as baseline)
total_time  = 0
total_flops = 0
all_scores  = []

for lang in langs:
    model = trained_models[lang]
    model.eval()

    # Tokenize test texts once
    x_texts = [str(t) for t in ds[f'{lang}_test']['combo']]
    enc = tokenizer(
        x_texts, max_length=MAX_LEN, padding='max_length',
        truncation=True, return_tensors='pt'
    )
    input_ids      = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)

    # Runtime measurement (10 runs)
    with torch.profiler.profile(with_flops=True) as p:
        begin = time.time()
        with torch.no_grad():
            for _ in range(10):
                logits = model(input_ids, attention_mask)
        total = time.time() - begin
    total_time  += total
    total_flops += sum(k.flops for k in p.key_averages()) / 1e9

    # Per-category scores
    lang_scores = evaluate_model(lang, model, ds, tokenizer)
    all_scores.extend(lang_scores)

scores_df = pd.DataFrame(all_scores)
print(scores_df.to_string(index=False))

  lang                category  precision   recall       f1
  java                 summary   0.901830 0.877023 0.889253
  java               Ownership   1.000000 1.000000 1.000000
  java                  Expand   0.333333 0.645570 0.439655
  java                   usage   0.904412 0.833898 0.867725
  java                 Pointer   0.696629 0.992000 0.818482
  java             deprecation   0.888889 0.800000 0.842105
  java                rational   0.258993 0.620690 0.365482
python                   Usage   0.871429 0.670330 0.757764
python              Parameters   0.811111 0.858824 0.834286
python        DevelopmentNotes   0.411765 0.437500 0.424242
python                  Expand   0.479452 0.686275 0.564516
python                 Summary   0.597701 0.852459 0.702703
 pharo Keyimplementationpoints   0.459459 0.607143 0.523077
 pharo                 Example   0.863636 0.853933 0.858757
 pharo        Responsibilities   0.500000 0.785714 0.611111
 pharo                  Intent   0.69230

## Step 9 — Final competition score

In [ ]:
max_avg_runtime = 5
max_avg_flops   = 5000

def competition_score(avg_f1, avg_runtime, avg_flops):
    return (
        0.6 * avg_f1
        + 0.2 * max((max_avg_runtime - avg_runtime) / max_avg_runtime, 0)
        + 0.2 * max((max_avg_flops   - avg_flops)   / max_avg_flops,   0)
    )

avg_f1      = scores_df['f1'].mean()
avg_runtime = total_time  / 10
avg_flops   = total_flops / 10

final_score = round(competition_score(avg_f1, avg_runtime, avg_flops), 4)

print('=== Results ===')
print(f'Average F1:      {avg_f1:.4f}')
print(f'Avg runtime (s): {avg_runtime:.4f}')
print(f'Avg GFLOPs:      {avg_flops:.2f}')
print(f'Competition score: {final_score}')

=== Results ===
Average F1:      0.6841
Avg runtime (s): 10.5641
Avg GFLOPs:      36948.18
Competition score: 0.4104


## Step 10 — Comparison table: baseline vs our model

In [ ]:
baseline = {
    'java':   {'summary': 0.879, 'Ownership': 1.0, 'Expand': 0.374, 'usage': 0.867,
               'Pointer': 0.861, 'deprecation': 0.778, 'rational': 0.356},
    'python': {'Usage': 0.674, 'Parameters': 0.719, 'DevelopmentNotes': 0.305,
               'Expand': 0.540, 'Summary': 0.672},
    'pharo':  {'Keyimplementationpoints': 0.600, 'Example': 0.881, 'Responsibilities': 0.681,
               'Intent': 0.783, 'Keymessages': 0.579, 'Collaborators': 0.167}
}

rows = []
for _, row in scores_df.iterrows():
    bl = baseline[row['lang']][row['category']]
    rows.append({
        'lang':     row['lang'],
        'category': row['category'],
        'baseline_f1': bl,
        'ours_f1':     round(row['f1'], 4),
        'delta':        round(row['f1'] - bl, 4)
    })

cmp = pd.DataFrame(rows)
print(cmp.to_string(index=False))
print(f"\nOverall avg F1 — Baseline: {cmp['baseline_f1'].mean():.4f}  |  Ours: {cmp['ours_f1'].mean():.4f}")

  lang                category  baseline_f1  ours_f1   delta
  java                 summary        0.879   0.8893  0.0103
  java               Ownership        1.000   1.0000  0.0000
  java                  Expand        0.374   0.4397  0.0657
  java                   usage        0.867   0.8677  0.0007
  java                 Pointer        0.861   0.8185 -0.0425
  java             deprecation        0.778   0.8421  0.0641
  java                rational        0.356   0.3655  0.0095
python                   Usage        0.674   0.7578  0.0838
python              Parameters        0.719   0.8343  0.1153
python        DevelopmentNotes        0.305   0.4242  0.1192
python                  Expand        0.540   0.5645  0.0245
python                 Summary        0.672   0.7027  0.0307
 pharo Keyimplementationpoints        0.600   0.5231 -0.0769
 pharo                 Example        0.881   0.8588 -0.0222
 pharo        Responsibilities        0.681   0.6111 -0.0699
 pharo                  

## Step 11 — Save models (optional)

In [ ]:
import os
os.makedirs('models', exist_ok=True)

for lang in langs:
    path = f'models/codebert_{lang}.pt'
    torch.save(trained_models[lang].state_dict(), path)
    print(f'Saved: {path}')

tokenizer.save_pretrained('models/tokenizer')
print('Tokenizer saved.')